# MLOps Internship Assessment: Wine Classification Workflow

This notebook demonstrates a complete end-to-end machine learning workflow on the classic UCI Wine dataset. It is designed to showcase how to go from exploratory analysis through model training and persistence. Your task as an intern candidate is to understand each step and then extend this work by exposing the trained model through a FastAPI service and packaging everything inside a Docker container.

## Dataset & Objective
- **Dataset**: UCI Wine recognition dataset (178 samples, 13 continuous features, 3 wine cultivars).
- **Goal**: Train a robust classifier that can predict a wine's cultivar based on its chemical analysis.
- **Context**: Everything you need lives in this repository (see `data/wine.csv`).
- **Deliverable**: After reviewing the workflow below, you will productionize the trained model via FastAPI and Docker. Instructions are at the end of this notebook.

## Environment Setup
Make sure you are working in a Python 3.9+ environment with the following packages installed:

```bash
pip install pandas numpy scikit-learn seaborn matplotlib joblib fastapi uvicorn[standard]
```

Feel free to create a virtual environment (via `venv`, `conda`, or `poetry`) before running the notebook.

In [ ]:
%matplotlib inline
import pathlib

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from joblib import dump

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)


## 1. Load & Inspect the Dataset

In [ ]:
data_path = pathlib.Path('..') / 'data' / 'wine.csv'
df = pd.read_csv(data_path)

df.head()

In [ ]:
print(f'Total rows: {len(df)}')
print(f'Columns: {list(df.columns)}')

df.describe().T

In [ ]:
df.info()

## 2. Exploratory Data Analysis

In [ ]:
class_counts = df['target'].value_counts().sort_index()
sns.barplot(x=class_counts.index, y=class_counts.values, palette='viridis')
plt.title('Class distribution (wine cultivar)')
plt.xlabel('Cultivar label')
plt.ylabel('Count')
plt.show()

In [ ]:
feature_columns = df.columns.drop('target')
df[feature_columns].hist(bins=15, figsize=(15, 10))
plt.suptitle('Feature distributions', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
correlation_matrix = df[feature_columns].corr()
sns.heatmap(correlation_matrix, cmap='RdBu_r', center=0)
plt.title('Feature correlation heatmap')
plt.show()

### Feature notes
- Some features are highly correlated (e.g. `flavanoids` and `od280_od315_of_diluted_wines`).
- Scaling features helps algorithms such as logistic regression converge faster.
- The target labels are balanced enough for standard cross-validation.

## 3. Prepare Data for Modeling

In [ ]:
X = df.drop(columns=['target'])
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)

## 4. Build & Tune a Classification Pipeline

In [ ]:
pipeline = Pipeline(
    steps=[
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=1000, multi_class='multinomial', solver='lbfgs'))
    ]
)

param_grid = {
    'clf__C': [0.1, 1.0, 10.0],
    'clf__penalty': ['l2']
}

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring='accuracy',
    cv=5,
    n_jobs=-1
)
grid_search.fit(X_train, y_train)

print(f"Best params: {grid_search.best_params_}")
print(f"CV accuracy: {grid_search.best_score_:.3f}")

## 5. Evaluate on the Test Set

In [ ]:
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted label')
plt.ylabel('True label')
plt.title('Confusion matrix')
plt.show()

## 6. Persist the Trained Model

In [ ]:
artifacts_dir = pathlib.Path('..') / 'artifacts'
artifacts_dir.mkdir(parents=True, exist_ok=True)
model_path = artifacts_dir / 'wine_classifier.joblib'
dump(best_model, model_path)

print(f"Model saved to {model_path.resolve()}")

sample = X_test.iloc[[0]]
print('Sample input (first row of test split):')
print(sample)
print(f"Predicted class: {best_model.predict(sample)[0]}")

## Next Steps: API & Containerization Challenge
You now have a trained model (`artifacts/wine_classifier.joblib`) and a repeatable workflow. To complete the internship assessment:

1. **FastAPI service**
   - Create a new FastAPI application that exposes an endpoint like `POST /predict` taking the 13 numeric features as input.
   - Load the serialized pipeline at startup and return the predicted class along with model metadata (e.g., training timestamp, accuracy).
2. **Validation**
   - Add simple request/response validation (Pydantic models) and at least one automated test (e.g., pytest) ensuring the endpoint works.
3. **Docker image**
   - Write a `Dockerfile` that installs the dependencies, copies the FastAPI app and artifacts, and launches Uvicorn.
   - Provide instructions or a shell script showing how to build and run the container locally.
4. **Documentation**
   - Update the project README with clear steps to recreate the environment, run tests, start the API, and exercise the endpoint.

Deliverables can live in a new `api/` directory with your FastAPI code, tests, and Docker assets. Keep commits atomic and describe the reasoning behind design choices. Good luck!